# Why a two-week shift wrecks a Markowitz book

You have a set of instruments, you estimate means and covariances on a trailing window, and you take the classical tangency portfolio

$$
w^\star \;\propto\; \hat\Sigma^{-1}\hat\mu.
$$

Slide the window by ten trading days. The economic world has barely moved. The weights have. This notebook shows that this is not a data bug — it is the estimator.

Two facts do all the work:

1. **Sample means are loud.** Merton (1980): you need decades of data to pin down $\mu$. A two-week slide already moves annualized $\hat\mu$ by several percent — the same order as the premia you are trying to rank.
2. **$\Sigma^{-1}$ is an error amplifier.** Equities share a factor. $\hat\Sigma$ is ill-conditioned. Noise that sits in the small-eigenvalue directions is multiplied by $1/\lambda_{\min}$ and becomes a leveraged long/short bet. Michaud (1989) called the optimizer an *error maximizer* for this reason.

The module `pytorch_katas.portfolio` builds a one-factor equity universe so the demonstration is deterministic. The same qualitative picture shows up on real ETFs.

In [1]:
from pathlib import Path

from pytorch_katas.portfolio import (
    TRADING_DAYS,
    TWO_WEEKS,
    allocate_window_pair,
    mean_shift_std,
    save_sensitivity_figures,
    simulate_equity_universe,
    summarize_pair,
)
from pytorch_katas.settings import DATA_DIR

## How loud is a two-week slide?

Write $\hat\mu_A$ for the sample mean on $[0, T)$ and $\hat\mu_B$ for the mean on $[h, T+h)$. Then

$$
\hat\mu_B - \hat\mu_A
= \frac{1}{T}\Big(\sum_{t=T}^{T+h-1} r_t - \sum_{t=0}^{h-1} r_t\Big).
$$

For iid returns the two endpoint blocks are independent, so

$$
\mathrm{std}(\hat\mu_B - \hat\mu_A) = \sigma\sqrt{2h}\,/\,T.
$$

With $\sigma = 1.5\%$ per day (about $24\%$ a year), $T = 252$, $h = 10$ that is already $\sim 7\%$ **annualized**. That is not estimation dust. It is larger than the typical gap between two equity expected returns.

In [2]:
daily_vol = 0.015
std_daily = mean_shift_std(daily_vol, TRADING_DAYS, TWO_WEEKS)
print(f"std of Δμ̂ (daily):      {std_daily:.5f}")
print(f"std of Δμ̂ (annualized): {std_daily * TRADING_DAYS:.2%}")

std of Δμ̂ (daily):      0.00027
std of Δμ̂ (annualized): 6.71%


## A two-week shift on a 12-name factor book

Twelve names, one market factor, betas from 0.75 to 1.25, true premia glued to the CAPM line. A year-long window, then the same window ten trading days later.

In [3]:
universe = simulate_equity_universe(n_days=750, n_assets=12, seed=0)
pair = allocate_window_pair(universe.returns, start=200)
print(summarize_pair(pair, universe.names))

Window 252 days, shifted by 10 trading days (~2 weeks).
Condition number of Σ̂ (window A): 54.3
Largest |Δμ| (annualized): 7.08%
Cross-sectional std of Δμ (annualized): 3.45%

rule           turnover  max |w| A  max |w| B   max |Δw|
tangency        302.6%     334.8%     398.0%      98.1%
gmv              11.5%      45.4%      49.9%       4.5%
ridge           104.5%     132.9%     133.8%      42.3%
shrink           25.5%      55.2%      52.9%      10.5%
long_only         0.0%     100.0%     100.0%       0.0%
equal             0.0%       8.3%       8.3%       0.0%

Annualized sample means, window A vs B:
  A01:  -7.05%  →   -8.85%  (Δ  -1.81%)
  A02:  -1.52%  →    4.11%  (Δ  +5.62%)
  A03:  11.11%  →   13.38%  (Δ  +2.26%)
  A04:  23.75%  →   22.64%  (Δ  -1.11%)
  A05:   9.31%  →    6.81%  (Δ  -2.50%)
  A06:   8.87%  →    5.09%  (Δ  -3.78%)
  A07:  10.25%  →    3.17%  (Δ  -7.08%)
  A08:   3.25%  →    5.40%  (Δ  +2.16%)
  A09:   0.60%  →    5.01%  (Δ  +4.41%)
  A10: -12.14%  →  -10.55%  (Δ

## What the optimizer does with that noise

Diagonalize $\hat\Sigma = Q\Lambda Q^\top$. Then

$$
\hat\Sigma^{-1}\hat\mu = Q\Lambda^{-1}Q^\top\hat\mu.
$$

Any component of $\hat\mu$ that lines up with a tiny eigenvalue — the near-arbitrage, high-correlation residual — is multiplied by $1/\lambda_{\min}$. That is why unconstrained tangency weights are leveraged and why they flip sign when the window slides.

Three things calm the book down without abandoning mean-variance:

- **Do not trust $\hat\mu$.** Shrink every mean toward the grand mean (James–Stein / Black–Litterman with a flat view).
- **Do not invert the sample covariance.** Ledoit–Wolf or a ridge $\hat\Sigma + \delta I$ kills the small eigenvalues.
- **Or drop $\mu$ entirely.** Global min-variance, risk parity, and $1/N$ are boring on purpose. DeMiguel, Garlappi & Uppal (2009): $1/N$ beats estimated Markowitz out of sample for this reason.

Long-only constraints help but do not fix it: the solution just jumps from one corner of the simplex to another.

In [4]:
fig_dir = Path(DATA_DIR) / "portfolio_figures"
paths = save_sensitivity_figures(universe, fig_dir)
for key, path in paths.items():
    print(f"{key:10s} {path}")

weights    /workspace/data/portfolio_figures/window_shift_weights.png
rolling    /workspace/data/portfolio_figures/rolling_allocation_heatmap.png
mechanism  /workspace/data/portfolio_figures/mean_shift_and_eigenvalues.png


Open the three PNGs in `data/portfolio_figures/`:

- `window_shift_weights.png` — same book, ten days later.
- `rolling_allocation_heatmap.png` — unconstrained tangency vs. shrinkage over the whole sample.
- `mean_shift_and_eigenvalues.png` — the two ingredients, $\Delta\hat\mu$ and $\kappa(\hat\Sigma)$.

The allocation did not change because the investment opportunity changed. It changed because classical Markowitz treats a noisy, two-week-sensitive statistic as if it were known.